# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
aviation_df = pd.read_csv(r"data\AviationData.csv",encoding="latin1",low_memory=False)
aviation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
df = aviation_df.loc[:,["Make","Model","Aircraft.damage","Aircraft.Category","Total.Fatal.Injuries","Total.Serious.Injuries","Total.Minor.Injuries","Total.Uninjured","Injury.Severity","Event.Date","Amateur.Built","Engine.Type","Weather.Condition","Number.of.Engines","Purpose.of.flight","Broad.phase.of.flight"]]
df["Event.Date"] = pd.to_datetime(df["Event.Date"])
df = df[(df["Event.Date"] >= "1983-01-01")&(df["Amateur.Built"] != "Yes")&(df["Aircraft.Category"]!="Helicopter")&(df["Make"]!= "Schleicher")&(df["Make"]!= "Texas Helicopter")&(df["Make"]!= "Brantly Helicopter")&(df["Make"]!= "G&c Helicopters")&(df["Make"]!= "Oregon Helicopters")&(df["Make"]!= "Westland Helicopters")&(df["Make"]!= "American Eurocopter")&(df["Make"]!= "Heli-eagle Inc.")&(df["Make"]!= "Piasecki Acft. Corp.")&(df["Make"]!= "Bell-continental Copters")&(df["Make"]!= "Bell-k Copter")&(df["Make"]!= "Bell-olympic Helicopters")&(df["Make"]!= "Bell-olympic Helicopters, Inc.")&(df["Make"]!= "Mbb")&(df["Make"]!= "Eurocopter")&(df["Make"]!= "Continental Copters")&(df["Make"]!= "Balloon Works")&(df["Make"]!= "Eagle Balloons")&(df["Make"]!= "Head Balloons, Inc.")&(df["Make"]!= "National Balloon")&(df["Make"]!= "Avian Balloon")]
df.dropna(subset=["Weather.Condition","Purpose.of.flight","Number.of.Engines","Total.Serious.Injuries","Total.Fatal.Injuries","Total.Uninjured","Model","Engine.Type","Aircraft.damage","Broad.phase.of.flight","Total.Minor.Injuries","Amateur.Built"],inplace=True)
df["Aircraft.Category"].fillna("Airplane",inplace=True)
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 39288 entries, 3601 to 63896
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Make                    39288 non-null  object        
 1   Model                   39288 non-null  object        
 2   Aircraft.damage         39288 non-null  object        
 3   Aircraft.Category       39288 non-null  object        
 4   Total.Fatal.Injuries    39288 non-null  float64       
 5   Total.Serious.Injuries  39288 non-null  float64       
 6   Total.Minor.Injuries    39288 non-null  float64       
 7   Total.Uninjured         39288 non-null  float64       
 8   Injury.Severity         39288 non-null  object        
 9   Event.Date              39288 non-null  datetime64[ns]
 10  Amateur.Built           39288 non-null  object        
 11  Engine.Type             39288 non-null  object        
 12  Weather.Condition       39288 non-null  object  

C:\Users\Ben\AppData\Local\Temp\ipykernel_16000\1804956109.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Aircraft.Category"].fillna("Airplane",inplace=True)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
df["total_passengers"] =  df[["Total.Fatal.Injuries","Total.Serious.Injuries","Total.Minor.Injuries","Total.Uninjured"]].sum(axis=1)
df["likelihood_serious_fatal"] = (df[["Total.Fatal.Injuries","Total.Serious.Injuries"]].sum(axis=1))/df["total_passengers"]

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
df["Destroyed?"] = np.where(df["Aircraft.damage"] == "Destroyed" ,1,0)

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
df = df[(df["Make"] != "unknown")&(df["Make"] != "Unknown")]
filtered_df = df[df["Make"].map(df["Make"].value_counts() <= 50 )]


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
models = df["Model"].dropna()
models = filtered_df.groupby(["Model","Make"]).value_counts()
filtered_df["id"] = filtered_df.groupby(["Model","Make"]).ngroup()
filtered_df.set_index("id",inplace=True)
filtered_df.head()

C:\Users\Ben\AppData\Local\Temp\ipykernel_16000\3668714420.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["id"] = filtered_df.groupby(["Model","Make"]).ngroup()


,Make,Model,Aircraft.damage,Aircraft.Category,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Injury.Severity,Event.Date,Amateur.Built,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight,total_passengers,likelihood_serious_fatal,Destroyed?
id,,,,,,,,,,,,,,,,,,,
452,Canadair,CL-600-1A11,Destroyed,Airplane,2.0,0.0,0.0,0.0,Fatal(2),1983-01-03,No,Turbo Fan,VMC,2.0,Executive/corporate,Approach,2.0,1.000000,1
1118,Teal,TSC-1A,Destroyed,Airplane,1.0,0.0,1.0,0.0,Fatal(1),1983-01-06,No,Reciprocating,UNK,1.0,Personal,Maneuvering,2.0,0.500000,1
154,Convair,580-11-A,Substantial,Airplane,1.0,1.0,0.0,31.0,Fatal(1),1983-01-09,No,Reciprocating,IMC,2.0,Unknown,Landing,33.0,0.060606,0
30,Israel Aircraft Industries,1124,Substantial,Airplane,0.0,0.0,0.0,8.0,Non-Fatal,1983-01-19,No,Turbo Fan,VMC,2.0,Executive/corporate,Takeoff,8.0,0.000000,0
160,"Smith, Ted Aerostar",600 AEROSTAR,Minor,Airplane,0.0,0.0,0.0,1.0,Incident,1983-01-24,No,Reciprocating,VMC,2.0,Unknown,Landing,1.0,0.000000,0


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39287 entries, 3601 to 63896
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Make                      39287 non-null  object        
 1   Model                     39287 non-null  object        
 2   Aircraft.damage           39287 non-null  object        
 3   Aircraft.Category         39287 non-null  object        
 4   Total.Fatal.Injuries      39287 non-null  float64       
 5   Total.Serious.Injuries    39287 non-null  float64       
 6   Total.Minor.Injuries      39287 non-null  float64       
 7   Total.Uninjured           39287 non-null  float64       
 8   Injury.Severity           39287 non-null  object        
 9   Event.Date                39287 non-null  datetime64[ns]
 10  Amateur.Built             39287 non-null  object        
 11  Engine.Type               39287 non-null  object        
 12  Weather.Condition   

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [9]:
filtered_df.to_csv("cleaned_airplane_data.csv")